# Задание №9. Создание модели для выявления фейковых новостей (классификация текстов новостей с помощью BiLSTM)

Данное задание представляет собой пошаговое руководство по созданию модели для классификации новостей на настоящие и фейковые (fake news detection) с использованием двунаправленной рекуррентной нейронной сети (BiLSTM). Вы познакомитесь с основными понятиями обработки естественного языка (NLP), научитесь подготавливать размеченный корпус новостей, строить и обучать модель, а также использовать её для предсказания достоверности новых текстов. В задании **обязательным требованием** является фиксация времени обучения модели – вы должны добавить в код соответствующие замеры и вывести результат. Задание адаптировано для выполнения в **Google Colab** (Jupyter Notebook). Все части работы (подготовка данных, обучение, сохранение, загрузка из репозитория и инференс) выполняются в одном ноутбуке.

---

## 1. Теоретическое введение: ключевые понятия выявления фейковых новостей

### 1.1. Задача обнаружения фейковых новостей
Фейковые новости (fake news) – это намеренно ложная или вводящая в заблуждение информация, распространяемая через новостные каналы или социальные сети. Задача автоматического выявления фейковых новостей заключается в бинарной классификации текстов: настоящие (real, 0) или фейковые (fake, 1). Это одна из актуальных задач NLP, помогающая бороться с дезинформацией.

### 1.2. Корпус данных
Для обучения необходим размеченный датасет, содержащий тексты новостей и метки достоверности. Популярные открытые датасеты: **LIAR**, **FakeNewsNet**, **ISOT Fake News**, **Kaggle Fake News** и др.

### 1.3. Токенизация (Tokenization)
Разбиение текста на минимальные единицы – токены (слова, знаки препинания). Для токенизации используем `Tokenizer` из Keras, который преобразует текст в последовательность целочисленных индексов.

### 1.4. Эмбеддинги (Embeddings)
Слой **Embedding** преобразует индексы токенов в плотные векторы фиксированной размерности. Эти векторы обучаются вместе с моделью и позволяют улавливать семантические и синтаксические сходства между словами.

### 1.5. Двунаправленный LSTM (Bidirectional LSTM)
В отличие от обычного LSTM, который обрабатывает последовательность только слева направо, двунаправленный LSTM состоит из двух независимых LSTM: один читает текст в прямом порядке, другой – в обратном. Их выходы объединяются, что позволяет модели учитывать контекст с обеих сторон. Это особенно полезно для выявления фейковых новостей, где важны как начало, так и конец текста.

### 1.6. Метрики качества
Для бинарной классификации используют:
- **Accuracy** – доля правильных ответов.
- **Precision (точность)** – доля истинно‑положительных среди всех предсказанных положительных.
- **Recall (полнота)** – доля найденных положительных примеров.
- **F1‑score** – гармоническое среднее precision и recall.
- **AUC‑ROC** – площадь под ROC‑кривой.

### 1.7. Фиксация времени обучения
Для оценки производительности и воспроизводимости эксперимента важно фиксировать время, затраченное на обучение модели. В коде необходимо использовать модуль `time` для замера длительности обучения и вывода результата.

---

## 2. Постановка задачи

Вам необходимо создать собственный репозиторий на GitHub, загрузить в него размеченный корпус новостей (файл `fake_news.csv`), затем построить и обучить модель BiLSTM для бинарной классификации новостей на настоящие и фейковые. **Обязательное требование:** в коде обучения модели нужно замерить время, затраченное на обучение, и вывести его в консоль (например, с помощью `time.time()`). После обучения вы сохраните модель, токенизатор и (при необходимости) кодировщик меток, загрузите их в тот же репозиторий. В финальной части ноутбука вы продемонстрируете, как загрузить эти файлы из репозитория и использовать их для предсказания достоверности новых новостей без повторного обучения. Весь код и отчёт должны быть оформлены в одном Jupyter Notebook.

Формат файла с данными: текстовый файл с разделителем `,` или `\t`, содержащий колонки `label` (real/fake) и `text` (текст новости). Пример строки из датасета ISOT:
```
"text","label"
"Washington, D.C. is the capital of the United States.",0
"BREAKING: Hillary Clinton arrested for treason!",1
```

---

## 3. Структура отчёта

Ваш отчёт должен содержать следующие разделы (в виде ячеек Markdown в ноутбуке):

1. **Титульный лист** (название работы, ФИО, группа, ссылка на репозиторий)
2. **Введение** (цель работы, краткое описание задачи выявления фейковых новостей)
3. **Теоретическая часть** (объяснение ключевых понятий: фейковые новости, токенизация, эмбеддинги, BiLSTM, метрики)
4. **Описание данных** (источник данных, статистика: количество примеров, распределение по классам, средняя длина текста; ссылка на файл в репозитории)
5. **Подготовка данных** (загрузка, предобработка, токенизация, паддинг, разделение на train/test)
6. **Построение модели** (архитектура BiLSTM, визуализация модели, summary)
7. **Обучение модели** (параметры обучения, использование early stopping, **фиксация времени обучения**, графики потерь и точности)
8. **Оценка качества** (расчёт accuracy, precision, recall, F1, матрица ошибок)
9. **Функция предсказания** (описание функции `predict_news`, демонстрация примеров)
10. **Сохранение модели и вспомогательных объектов** (код сохранения модели, токенизатора; загрузка файлов в репозиторий)
11. **Загрузка модели из репозитория и быстрый инференс** (код загрузки через raw‑ссылку, повторное использование для предсказания)
12. **Выводы** (что получилось, какие были трудности, возможные улучшения)
13. **Список использованных источников**
14. **Приложение** (полный код с комментариями)

---

## 4. Источники данных для выявления фейковых новостей

### 4.1. Открытые датасеты

| Источник | Описание | Ссылка |
|----------|----------|--------|
| **ISOT Fake News** | 44 000+ новостей, сбалансированный | https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets |
| **LIAR Dataset** | Короткие утверждения с правдивостью (6 уровней) | https://www.cs.ucsb.edu/~william/liar_dataset.html |
| **FakeNewsNet** | Новости с фактчекингом (PolitiFact, GossipCop) | https://github.com/KaiDMML/FakeNewsNet |
| **Kaggle: Fake News** | 20 000+ новостей, метки real/fake | https://www.kaggle.com/c/fake-news/data |
| **Getting Real about Fake News** | 20 000 статей | https://www.kaggle.com/datasets/mrisdal/fake-news |

### 4.2. Синтетический датасет (для тренировки)

Если у вас нет возможности скачать реальные данные, вы можете сгенерировать синтетический датасет с помощью простого шаблонизатора. Однако для реального выполнения задания рекомендуется использовать реальный датасет.

```python
import pandas as pd
import random

# Шаблоны настоящих новостей (реалистичные)
real_templates = [
    "Президент подписал закон о поддержке малого бизнеса.",
    "Ученые обнаружили новый вид динозавров в Южной Америке.",
    "Компания объявила о запуске новой линейки смартфонов.",
    "Завтра в городе ожидается дождь и ветер до 15 м/с.",
    "Цены на нефть выросли на 3% после заседания ОПЕК+."
]
# Шаблоны фейковых новостей (сенсационные, преувеличенные)
fake_templates = [
    "СРОЧНО: Инопланетяне высадились в центре Москвы!",
    "Чиновники украли 10 миллиардов рублей из бюджета – доказано!",
    "Это лекарство полностью излечивает рак за 2 дня.",
    "Найден способ вечной жизни – подробности внутри.",
    "Правительство скрывает правду о глобальном потеплении."
]

data = []
for _ in range(1000):
    data.append([random.choice(real_templates), 0])
for _ in range(1000):
    data.append([random.choice(fake_templates), 1])

random.shuffle(data)
df = pd.DataFrame(data, columns=["text", "label"])
df.to_csv("fake_news_data.csv", index=False)
```

### 4.3. Требования к объёму корпуса
- **Минимальный объём**: не менее 2 000 примеров, сбалансированных или с естественным распределением.
- **Максимальный объём**: в Google Colab ограничение по оперативной памяти; до 100 000 примеров – комфортно.

### 4.4. Создание репозитория и загрузка данных
1. Зарегистрируйтесь на [GitHub](https://github.com).
2. Создайте новый публичный репозиторий с названием, например, `fake-news-detector`.
3. Загрузите в репозиторий файл с данными (например, `fake_news_data.csv`).
4. Получите **raw‑ссылку** на файл (открыть файл → Raw → скопировать URL). Пример:  
   `https://raw.githubusercontent.com/ваш_логин/fake-news-detector/main/fake_news_data.csv`

---

## 5. Пошаговое выполнение задания в Google Colab

### 5.1. Подготовка окружения

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import pickle
import requests
import io
```

### 5.2. Загрузка данных из репозитория

```python
# Вставьте вашу ссылку
url_data = "https://raw.githubusercontent.com/ваш_логин/fake-news-detector/main/fake_news_data.csv"

response = requests.get(url_data)
df = pd.read_csv(io.StringIO(response.text))

print(f"Загружено примеров: {len(df)}")
print(df['label'].value_counts())
df.head()
```

### 5.3. Предобработка текста и подготовка данных

```python
def clean_text(text):
    text = str(text).lower()
    # Можно добавить более сложную очистку: удаление пунктуации, стоп-слов и т.д.
    return text

df['clean_text'] = df['text'].apply(clean_text)

# Параметры токенизации
VOCAB_SIZE = 15000  # максимальный размер словаря
MAX_LENGTH = 200    # максимальная длина текста (в словах)

# Токенизация
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'])
sequences = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Метки уже числовые (0/1)
y = df['label'].values

print(f"Форма X: {X.shape}")
print(f"Распределение меток: {np.bincount(y)}")
```

### 5.4. Разделение на обучающую и тестовую выборки

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающих примеров: {len(X_train)}")
print(f"Тестовых примеров: {len(X_test)}")
```

### 5.5. Построение модели BiLSTM

```python
EMBEDDING_DIM = 100
LSTM_UNITS = 64
DROPOUT_RATE = 0.3

model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    Bidirectional(LSTM(LSTM_UNITS, dropout=DROPOUT_RATE)),
    Dense(64, activation='relu'),
    Dropout(DROPOUT_RATE),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
```

### 5.6. Обучение модели с фиксацией времени

**Обязательное требование:** добавьте код для измерения времени обучения и выведите его.

```python
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("Начало обучения...")
start_time = time.time()

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

end_time = time.time()
training_time = end_time - start_time
print(f"\nОбучение завершено за {training_time:.2f} секунд ({(training_time/60):.2f} минут)")
```

### 5.7. Визуализация процесса обучения

```python
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери (Loss)')

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Точность (Accuracy)')
plt.show()
```

### 5.8. Оценка качества модели

```python
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-score: {f1:.4f}")

# Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Матрица ошибок')
plt.ylabel('Истинный класс')
plt.xlabel('Предсказанный класс')
plt.show()
```

### 5.9. Функция предсказания

```python
def predict_news(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    prob = model.predict(padded, verbose=0)[0][0]
    label = "FAKE" if prob > 0.5 else "REAL"
    return label, prob

# Примеры
test_news = [
    "Ученые подтвердили, что изменение климата – реальная угроза.",
    "СРОЧНО: Земля столкнется с гигантским астероидом завтра!",
    "Президент подписал закон о новых налоговых льготах."
]

for news in test_news:
    label, prob = predict_news(news)
    print(f"Текст: {news}")
    print(f"Вердикт: {label} (вероятность фейка: {prob:.4f})\n")
```

### 5.10. Сохранение модели и вспомогательных объектов

```python
model.save('fake_news_model.h5')
print("Модель сохранена в fake_news_model.h5")

with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Токенизатор сохранён в tokenizer.pickle")
```

### 5.11. Загрузка файлов в репозиторий

1. Скачайте файлы `fake_news_model.h5` и `tokenizer.pickle` на свой компьютер.
2. В своём репозитории на GitHub загрузите эти файлы.
3. Получите raw‑ссылки на каждый файл (формат `https://github.com/ваш_логин/fake-news-detector/raw/main/...`).

### 5.12. Загрузка модели из репозитория и быстрый инференс

```python
url_model = "https://github.com/ваш_логин/fake-news-detector/raw/main/fake_news_model.h5"
url_tokenizer = "https://github.com/ваш_логин/fake-news-detector/raw/main/tokenizer.pickle"

!wget -O fake_news_model.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}

loaded_model = load_model('fake_news_model.h5')
with open('tokenizer.pickle', 'rb') as f:
    loaded_tokenizer = pickle.load(f)

def predict_loaded(text):
    cleaned = clean_text(text)
    seq = loaded_tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    prob = loaded_model.predict(padded, verbose=0)[0][0]
    return "FAKE" if prob > 0.5 else "REAL", prob

# Пример
for news in test_news:
    label, prob = predict_loaded(news)
    print(f"Текст: {news}")
    print(f"Вердикт (загруженная модель): {label} (вероятность фейка: {prob:.4f})\n")
```

---



## 6. Код с загруженной моделью для быстрого запуска

```python
url_model = "https://github.com/ваш_логин/fake-news-detector/raw/main/fake_news_model.h5"
url_tokenizer = "https://github.com/ваш_логин/fake-news-detector/raw/main/tokenizer.pickle"

!wget -O fake_news_model.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}

import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

loaded_model = load_model('fake_news_model.h5')
with open('tokenizer.pickle', 'rb') as f: tokenizer = pickle.load(f)

MAX_LENGTH = 200

def predict_news(text):
    cleaned = str(text).lower()
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    prob = loaded_model.predict(padded, verbose=0)[0][0]
    return "FAKE" if prob > 0.5 else "REAL", prob

# Пример
print(predict_news("Ученые нашли лекарство от всех болезней!"))
```

---

## 7. Задания для студентов

### Задание 1. Подготовка репозитория и данных
- Создайте публичный репозиторий на GitHub.
- Выберите один из предложенных датасетов (рекомендуется ISOT Fake News или Kaggle Fake News) и загрузите его в репозиторий под именем `fake_news_data.csv`.
- Получите raw‑ссылку на файл.

### Задание 2. Подготовка данных в ноутбуке
- Загрузите данные из репозитория по ссылке.
- Выведите статистику: количество примеров, распределение по классам, гистограмму длин текстов.
- Выполните очистку текста (приведение к нижнему регистру, удаление лишних символов).
- Выполните токенизацию и паддинг последовательностей.
- Разделите данные на обучающую и тестовую выборки (80/20) со стратификацией.

### Задание 3. Построение модели
- Постройте модель с двунаправленным LSTM (Bidirectional) и Dropout.
- Выведите summary модели.
- Скомпилируйте модель с оптимизатором 'adam' и функцией потерь 'binary_crossentropy'.

### Задание 4. Обучение модели с фиксацией времени (обязательно)
- Обучите модель с использованием EarlyStopping (patience=5) на 30 эпохах.
- **Обязательно** добавьте код для измерения времени обучения и выведите результат в консоль.
- Постройте графики потерь и точности на обучающей и валидационной выборках.

### Задание 5. Оценка качества
- Предскажите классы на тестовой выборке.
- Выведите classification report (precision, recall, F1) и матрицу ошибок.
- Проанализируйте, какие типы ошибок допускает модель (ложноположительные и ложноотрицательные).

### Задание 6. Функция предсказания
- Реализуйте функцию `predict_news`, которая принимает текст и возвращает метку (REAL/FAKE) и вероятность фейка.
- Протестируйте на 3-5 собственных примерах (включая явно фейковые и реальные новости).

### Задание 7. Сохранение и загрузка в репозиторий
- Сохраните модель и токенизатор.
- Загрузите эти файлы в свой репозиторий.
- Получите raw‑ссылки на каждый файл.

### Задание 8. Загрузка модели из репозитория и быстрый инференс
- В отдельной ячейке загрузите файлы из репозитория.
- Восстановите модель и выполните предсказание для тех же примеров, что и в задании 6.
- Убедитесь, что результаты совпадают.

### Задание 9. Анализ результатов
- Оцените, насколько хорошо модель отличает фейковые новости от реальных.
- Предложите способы улучшения модели (например, увеличение размера словаря, использование предобученных эмбеддингов, добавление дополнительных признаков).

### Задание 10*. Дополнительно (по желанию)
- Попробуйте использовать обычный LSTM вместо Bidirectional и сравните результаты.
- Добавьте в предобработку удаление стоп-слов и лемматизацию. Сравните результаты.
- Визуализируйте эмбеддинги слов с помощью t-SNE.

---

## 8. Заключение

В ходе выполнения этого задания вы:
- создали собственный репозиторий на GitHub и научились загружать туда файлы;
- познакомились с задачей обнаружения фейковых новостей и подходами к её решению;
- подготовили размеченные данные, выполнили токенизацию и паддинг;
- построили и обучили двунаправленную LSTM‑модель для бинарной классификации;
- **обязательно зафиксировали время обучения модели**;
- оценили качество модели с помощью метрик классификации;
- освоили сохранение модели и токенизатора, а затем их загрузку из репозитория для повторного использования.

Полученные навыки являются основой для создания систем автоматической проверки достоверности информации, что крайне важно в современном информационном пространстве.
